# 03 - Models Comparison

This notebook trains, evaluates, and compares five different machine learning algorithms on the Stage 4 advanced feature-engineered dataset:
1. **Linear Regression** (Baseline linear model)
2. **Random Forest** (Parallel bagging ensemble)
3. **XGBoost** (Gradient Boosting with fast histograms)
4. **LightGBM** (Leaf-wise Gradient Boosting)
5. **CatBoost** (Ordered Gradient Boosting)

All models are optimized to minimize target log sales `log1p(Sales)` to align directly with the primary competition metric: **RMSPE**.

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import time
import sys
import os

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

# Append source directory to path
sys.path.append(os.path.abspath("../src"))
from preprocessing import prepare_data
from evaluate import evaluate_predictions

## 2. Load Stage 4 Data

We load the clean, target-encoded, and non-leak temporal features.

In [ ]:
train_path = "../data/raw/train.csv"
store_path = "../data/raw/store.csv"

X_train, y_train, X_val, y_val = prepare_data(train_path, store_path, stage=4)
y_train_log = np.log1p(y_train)

## 3. Train and Compare Models

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=30, max_depth=12, random_state=42, n_jobs=-1),
    "XGBoost": XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, tree_method='hist', n_jobs=-1),
    "LightGBM": LGBMRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1, verbosity=-1),
    "CatBoost": CatBoostRegressor(iterations=100, depth=6, learning_rate=0.1, random_state=42, thread_count=-1, verbose=False)
}

results = []

for name, model in models.items():
    print(f"Training {name}...")
    start_time = time.time()
    model.fit(X_train, y_train_log)
    elapsed_time = time.time() - start_time
    
    # Predict & Inverse Transform
    y_pred_log = model.predict(X_val)
    y_pred = np.expm1(y_pred_log)
    y_pred = np.clip(y_pred, 0, None)
    
    # Evaluate
    metrics = evaluate_predictions(y_val, y_pred)
    results.append({
        "Model": name,
        "RMSE": round(metrics["RMSE"], 2),
        "RMSPE (%)": round(metrics["RMSPE"], 2),
        "MAE": round(metrics["MAE"], 2),
        "R2": round(metrics["R2"], 4),
        "Train Time (s)": round(elapsed_time, 2)
    })

## 4. Summary Table

Let's output the final comparison DataFrame.

In [ ]:
df_results = pd.DataFrame(results)
df_results